# Historical Data Evaluation

Notebook dedicado a avalia??o dos melhores modelos, m?tricas finais e simula??o de estrat?gia.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.historical_loader import load_historical_asset_dataframe
from src.data.historical_preprocessing import prepare_historical_market_dataframe
from src.features.historical_features import DEFAULT_HISTORICAL_FEATURES, build_log_diff_dataset, create_multi_period_targets
from src.models.historical_training import (
    build_classification_metrics_frame,
    evaluate_best_model_configs,
    prepare_lagged_modeling_data,
    simulate_strategy,
    tune_models_with_cv,
    build_default_param_spaces,
)


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
PERIODS = (1, 3, 7, 14, 30)
THRESHOLD_MULTIPLIER = 0.75
MAX_LAGS = 20
TRANSFORMED_FEATURE_COLUMNS = [
    'log_diff_open',
    'log_diff_high',
    'log_diff_low',
    'log_diff_close',
    'log_diff_volume_btc',
    'log_diff_volume_usdt',
    'log_diff_tradecount',
]
MODEL_SPACES = build_default_param_spaces()


In [ ]:
raw_btc_hist = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
btc_hist = prepare_historical_market_dataframe(raw_btc_hist)
btc_hist_transformed = build_log_diff_dataset(btc_hist, feature_columns=DEFAULT_HISTORICAL_FEATURES, keep_close_column=True)
df_with_targets = create_multi_period_targets(btc_hist_transformed, periods=PERIODS, threshold_multiplier=THRESHOLD_MULTIPLIER)


In [ ]:
results_by_horizon = {}
for horizon in PERIODS:
    target_column = f'target_{horizon}d'
    modeling_data = prepare_lagged_modeling_data(df_with_targets, TRANSFORMED_FEATURE_COLUMNS, MAX_LAGS, target_column)
    results_by_horizon[horizon] = tune_models_with_cv(
        modeling_data,
        TRANSFORMED_FEATURE_COLUMNS,
        target_column,
        MAX_LAGS,
        MODEL_SPACES,
    )
results_by_horizon


In [ ]:
evaluation_results = {}
for horizon in PERIODS:
    target_column = f'target_{horizon}d'
    modeling_data = prepare_lagged_modeling_data(df_with_targets, TRANSFORMED_FEATURE_COLUMNS, MAX_LAGS, target_column)
    evaluation_results[horizon] = evaluate_best_model_configs(
        modeling_data,
        TRANSFORMED_FEATURE_COLUMNS,
        {horizon: results_by_horizon[horizon]},
    )[horizon]
evaluation_results


In [ ]:
for horizon, result in evaluation_results.items():
    plt.figure(figsize=(6, 4))
    sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Horizon {horizon}d')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


In [ ]:
classification_metrics = build_classification_metrics_frame(df_with_targets, TRANSFORMED_FEATURE_COLUMNS, results_by_horizon)
classification_metrics


In [ ]:
classification_metrics.to_csv(PROJECT_ROOT / 'classification_metrics.csv', index=False)


In [ ]:
strategy_horizons = (3, 7, 30)
strategy_results = [simulate_strategy(df_with_targets, horizon, TRANSFORMED_FEATURE_COLUMNS, results_by_horizon) for horizon in strategy_horizons]

plt.figure(figsize=(16, 6))
for result in strategy_results:
    plt.plot(result.curve_strategy, label=f'Strategy t+{result.horizon}')
    plt.plot(result.curve_buy_hold, linestyle='--', label=f'Buy & Hold t+{result.horizon}')
plt.title('Curva de Capital Simulada por Horizonte')
plt.xlabel('Per?odos (test set)')
plt.ylabel('Capital acumulado')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame([
    {
        'horizon': result.horizon,
        'strategy_return': result.total_return_strategy,
        'buy_hold_return': result.total_return_buy_hold,
        'accuracy': result.accuracy,
    }
    for result in strategy_results
])
